# Fourier Analysis, Linear Regression, and Lomb–Scargle  
## A Python Demo for TESS Light Curve Research

This notebook explains three ideas that appear often in astrophysics data analysis:

1. **Fourier analysis**: how a periodic signal can be described using sine and cosine waves.
2. **Linear regression**: how we fit the best straight line to data.
3. **Lomb–Scargle periodogram**: how we find periods in unevenly sampled light curves, such as real astronomical data.

The examples are written for someone who has completed **AP Calculus BC** and knows basic Python.

## 0. Setup

We will use:

- `numpy` for arrays and math
- `matplotlib` for plots
- `scipy` for a basic Lomb–Scargle implementation

If `scipy` is missing, install it with:

```bash
pip install scipy
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lombscargle

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

np.random.seed(42)

# Part 1 — Fourier Analysis for Periodic Signals

## 1.1 Why Fourier analysis matters for light curves

A **light curve** is a plot of brightness versus time.

Many variable stars brighten and dim in repeating patterns. For example:

- an eclipsing binary may dim every time one star passes in front of the other
- a pulsating star may expand and contract periodically
- a rotating spotted star may show brightness changes as dark spots rotate in and out of view

Fourier analysis helps us answer this question:

> What frequencies or periods are hidden inside the brightness measurements?

A simple periodic signal can be written as a sine wave:

\[
y(t) = A\sin(2\pi f t + \phi)
\]

where:

- \(A\) is amplitude
- \(f\) is frequency, measured in cycles per day if time is in days
- \(P = 1/f\) is period
- \(\phi\) is phase shift

## 1.2 Create a simple periodic signal

Suppose a star has a period of 2 days.  
That means its frequency is:

\[
f = \frac{1}{P} = \frac{1}{2} = 0.5 \text{ cycles/day}
\]

In [ ]:
# Time in days
t = np.linspace(0, 10, 500)

true_period = 2.0
true_frequency = 1 / true_period

# A clean sine-wave light curve
flux = 1.0 + 0.1 * np.sin(2 * np.pi * true_frequency * t)

plt.plot(t, flux)
plt.xlabel("Time [days]")
plt.ylabel("Normalized Flux")
plt.title("Simple Periodic Light Curve: Period = 2 days")
plt.show()

## 1.3 Add noise, like real measurements

Real TESS light curves are not perfect. They include:

- photon noise
- instrumental effects
- data gaps
- possible long-term trends

Let us add random noise.

In [ ]:
noise = np.random.normal(0, 0.02, size=len(t))
flux_noisy = flux + noise

plt.scatter(t, flux_noisy, s=10, alpha=0.7)
plt.plot(t, flux, linewidth=2, label="True signal")
plt.xlabel("Time [days]")
plt.ylabel("Normalized Flux")
plt.title("Noisy Periodic Light Curve")
plt.legend()
plt.show()

## 1.4 Fourier idea: test many frequencies

The core idea is simple:

> Try many sine/cosine waves and measure which frequency matches the data best.

For a perfect evenly sampled signal, the Fast Fourier Transform, or FFT, is commonly used.

The FFT tells us how much power exists at different frequencies.

In [ ]:
# Remove the mean so the constant brightness level does not dominate
flux_centered = flux_noisy - np.mean(flux_noisy)

# Time spacing
dt = t[1] - t[0]

# FFT frequencies and amplitudes
fft_values = np.fft.rfft(flux_centered)
fft_freqs = np.fft.rfftfreq(len(t), d=dt)

fft_power = np.abs(fft_values) ** 2

plt.plot(fft_freqs, fft_power)
plt.xlim(0, 3)
plt.xlabel("Frequency [cycles/day]")
plt.ylabel("FFT Power")
plt.title("Fourier Power Spectrum")
plt.axvline(true_frequency, linestyle="--", label=f"True frequency = {true_frequency:.2f}")
plt.legend()
plt.show()

best_fft_frequency = fft_freqs[np.argmax(fft_power[1:]) + 1]
best_fft_period = 1 / best_fft_frequency

print(f"Best FFT frequency: {best_fft_frequency:.3f} cycles/day")
print(f"Best FFT period: {best_fft_period:.3f} days")

## 1.5 Important interpretation

The strongest peak appears near the true frequency:

\[
f = 0.5 \text{ cycles/day}
\]

which corresponds to:

\[
P = 2.0 \text{ days}
\]

This is the main idea behind frequency analysis:

> A periodic signal leaves a peak in frequency space.

For real TESS light curves, we often care more about **period** than frequency, because period connects directly to the physical behavior of the star.

## 1.6 A signal with multiple frequencies

Some light curves are not simple sine waves. They may contain harmonics or multiple periodic effects.

A signal can be built by adding sine waves:

\[
y(t) = A_1\sin(2\pi f_1 t) + A_2\sin(2\pi f_2 t)
\]

This is why Fourier analysis is powerful: it decomposes a complicated signal into simpler frequency components.

In [ ]:
f1 = 0.5   # period 2 days
f2 = 1.5   # period 0.667 days

multi_flux = (
    1.0
    + 0.10 * np.sin(2 * np.pi * f1 * t)
    + 0.04 * np.sin(2 * np.pi * f2 * t)
    + np.random.normal(0, 0.02, size=len(t))
)

plt.scatter(t, multi_flux, s=10, alpha=0.7)
plt.xlabel("Time [days]")
plt.ylabel("Normalized Flux")
plt.title("Light Curve with Two Frequency Components")
plt.show()

multi_centered = multi_flux - np.mean(multi_flux)
fft_values_multi = np.fft.rfft(multi_centered)
fft_power_multi = np.abs(fft_values_multi) ** 2

plt.plot(fft_freqs, fft_power_multi)
plt.xlim(0, 4)
plt.xlabel("Frequency [cycles/day]")
plt.ylabel("FFT Power")
plt.title("Fourier Power Spectrum with Two Peaks")
plt.axvline(f1, linestyle="--", label="f1 = 0.5")
plt.axvline(f2, linestyle="--", label="f2 = 1.5")
plt.legend()
plt.show()

# Part 2 — Linear Regression

## 2.1 What linear regression does

Linear regression fits a model of the form:

\[
y = mx + b
\]

where:

- \(m\) is the slope
- \(b\) is the intercept

The goal is to choose \(m\) and \(b\) so that the line is close to the data points.

For each data point, the model makes a prediction:

\[
\hat{y}_i = mx_i + b
\]

The error, or residual, is:

\[
r_i = y_i - \hat{y}_i
\]

Linear regression usually minimizes the sum of squared residuals:

\[
\sum_i (y_i - \hat{y}_i)^2
\]

This is called **least squares**.

## 2.2 Simple example: fitting a straight line

Suppose we have data that roughly follows:

\[
y = 2x + 1
\]

but with noise.

In [ ]:
x = np.linspace(0, 10, 30)
true_m = 2.0
true_b = 1.0

y = true_m * x + true_b + np.random.normal(0, 2.0, size=len(x))

plt.scatter(x, y)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Noisy Linear Data")
plt.show()

## 2.3 Fit the line using `numpy.polyfit`

`np.polyfit(x, y, 1)` fits a degree-1 polynomial:

\[
y = mx + b
\]

In [ ]:
estimated_m, estimated_b = np.polyfit(x, y, 1)

y_pred = estimated_m * x + estimated_b

plt.scatter(x, y, label="Data")
plt.plot(x, y_pred, linewidth=2, label="Best-fit line")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear Regression Fit")
plt.legend()
plt.show()

print(f"Estimated slope m: {estimated_m:.3f}")
print(f"Estimated intercept b: {estimated_b:.3f}")

## 2.4 Why least squares uses squared errors

The residuals can be positive or negative. If we simply added them, they could cancel out.

Squaring residuals has two effects:

1. all errors become positive
2. larger errors are punished more strongly

This is why least squares tries to find a line that balances the errors across all points.

In [ ]:
residuals = y - y_pred
sum_squared_error = np.sum(residuals ** 2)

plt.scatter(x, residuals)
plt.axhline(0, linewidth=2)
plt.xlabel("x")
plt.ylabel("Residual = actual - predicted")
plt.title("Residuals from Linear Regression")
plt.show()

print(f"Sum of squared errors: {sum_squared_error:.3f}")

## 2.5 Linear regression as a building block for Lomb–Scargle

This is the key connection:

> Lomb–Scargle is like doing many tiny regression problems.

For each trial frequency, Lomb–Scargle asks:

> If I fit a sine wave and cosine wave at this frequency, how well do they explain the data?

The model is approximately:

\[
y(t) = a\sin(2\pi f t) + b\cos(2\pi f t) + c
\]

For each frequency \(f\), the algorithm finds the best \(a\), \(b\), and \(c\), then measures how good the fit is.

The best frequency is the one that gives the strongest improvement over a flat model.

# Part 3 — Lomb–Scargle Frequency Analysis

## 3.1 Why FFT is not always enough for astronomy

The FFT works best when data are:

- evenly spaced in time
- continuous
- without large gaps

But real astronomical light curves often have:

- irregular time sampling
- missing observations
- data gaps
- multiple observing sectors

TESS data can be close to regular inside one sector, but gaps and quality cuts still matter.

The **Lomb–Scargle periodogram** is designed for unevenly sampled data.

## 3.2 Create an unevenly sampled light curve

We will create a simulated light curve with a true period of 1.75 days, then randomly remove many observations.

In [ ]:
# Start with regular time points
t_regular = np.linspace(0, 20, 800)

true_period_ls = 1.75
true_frequency_ls = 1 / true_period_ls

flux_regular = (
    1.0
    + 0.08 * np.sin(2 * np.pi * true_frequency_ls * t_regular + 0.5)
    + np.random.normal(0, 0.02, size=len(t_regular))
)

# Randomly keep only part of the data to mimic gaps/irregular sampling
keep = np.random.rand(len(t_regular)) > 0.35

# Add a larger observing gap
gap = (t_regular > 8) & (t_regular < 11)
keep = keep & (~gap)

t_irregular = t_regular[keep]
flux_irregular = flux_regular[keep]

plt.scatter(t_irregular, flux_irregular, s=10, alpha=0.7)
plt.xlabel("Time [days]")
plt.ylabel("Normalized Flux")
plt.title("Unevenly Sampled Simulated Light Curve")
plt.show()

print(f"Number of regular points: {len(t_regular)}")
print(f"Number of kept irregular points: {len(t_irregular)}")

## 3.3 Lomb–Scargle periodogram

We choose a grid of trial frequencies.  
For each frequency, Lomb–Scargle computes a power value.

Higher power means:

> A sinusoidal model at this frequency explains the data better.

Important detail: `scipy.signal.lombscargle` expects **angular frequency**:

\[
\omega = 2\pi f
\]

In [ ]:
# Center the flux
y_irregular = flux_irregular - np.mean(flux_irregular)

# Frequency grid in cycles/day
min_frequency = 0.05
max_frequency = 5.0
frequencies = np.linspace(min_frequency, max_frequency, 10000)

# Convert to angular frequency
angular_frequencies = 2 * np.pi * frequencies

# Lomb-Scargle power
power = lombscargle(t_irregular, y_irregular, angular_frequencies, normalize=True)

best_index = np.argmax(power)
best_frequency = frequencies[best_index]
best_period = 1 / best_frequency

plt.plot(frequencies, power)
plt.axvline(true_frequency_ls, linestyle="--", label=f"True frequency = {true_frequency_ls:.3f}")
plt.axvline(best_frequency, linestyle=":", label=f"Best frequency = {best_frequency:.3f}")
plt.xlabel("Frequency [cycles/day]")
plt.ylabel("Lomb–Scargle Power")
plt.title("Lomb–Scargle Periodogram")
plt.legend()
plt.show()

print(f"True period: {true_period_ls:.4f} days")
print(f"Best Lomb-Scargle period: {best_period:.4f} days")
print(f"True frequency: {true_frequency_ls:.4f} cycles/day")
print(f"Best Lomb-Scargle frequency: {best_frequency:.4f} cycles/day")

## 3.4 Periodogram in period space

Astronomers often prefer period instead of frequency.

Since:

\[
P = \frac{1}{f}
\]

we can plot power against period.

In [ ]:
periods = 1 / frequencies

plt.plot(periods, power)
plt.axvline(true_period_ls, linestyle="--", label=f"True period = {true_period_ls:.2f}")
plt.axvline(best_period, linestyle=":", label=f"Best period = {best_period:.2f}")
plt.xlim(0, 5)
plt.xlabel("Period [days]")
plt.ylabel("Lomb–Scargle Power")
plt.title("Lomb–Scargle Power vs Period")
plt.legend()
plt.show()

## 3.5 Phase folding

Once we find a likely period, we can **phase fold** the light curve.

The phase is:

\[
\text{phase} = \left(\frac{t}{P}\right) \bmod 1
\]

This puts all cycles on top of each other.

If the period is correct, the pattern should line up.

In [ ]:
phase = (t_irregular / best_period) % 1

plt.scatter(phase, flux_irregular, s=10, alpha=0.7)
plt.xlabel("Phase")
plt.ylabel("Normalized Flux")
plt.title("Phase-Folded Light Curve Using Best Lomb–Scargle Period")
plt.show()

## 3.6 Show two cycles for easier visualization

Many astronomy papers show phase from 0 to 2 instead of 0 to 1.  
This repeats the same folded curve twice and makes the shape easier to see.

In [ ]:
phase_two_cycles = np.concatenate([phase, phase + 1])
flux_two_cycles = np.concatenate([flux_irregular, flux_irregular])

plt.scatter(phase_two_cycles, flux_two_cycles, s=10, alpha=0.7)
plt.xlabel("Phase")
plt.ylabel("Normalized Flux")
plt.title("Phase-Folded Light Curve, Shown Over Two Cycles")
plt.show()

## 3.7 Lomb–Scargle as repeated sine/cosine regression

Here is a simplified manual version of the idea.

For each trial frequency \(f\), fit this model:

\[
y(t) = a\sin(2\pi f t) + b\cos(2\pi f t) + c
\]

This is linear regression because \(a\), \(b\), and \(c\) are linear coefficients.

The frequency itself is not linear, so we test many possible frequencies.

In [ ]:
def fit_sine_cosine_model(t, y, frequency):
    '''
    Fit y = a*sin(2*pi*f*t) + b*cos(2*pi*f*t) + c
    using least squares.
    '''
    X = np.column_stack([
        np.sin(2 * np.pi * frequency * t),
        np.cos(2 * np.pi * frequency * t),
        np.ones_like(t)
    ])

    coefficients, residuals, rank, singular_values = np.linalg.lstsq(X, y, rcond=None)
    y_fit = X @ coefficients

    sse = np.sum((y - y_fit) ** 2)
    return coefficients, y_fit, sse

# Compare a wrong frequency and the best frequency
wrong_frequency = 1.2

coef_wrong, fit_wrong, sse_wrong = fit_sine_cosine_model(t_irregular, y_irregular, wrong_frequency)
coef_best, fit_best, sse_best = fit_sine_cosine_model(t_irregular, y_irregular, best_frequency)

print(f"SSE at wrong frequency {wrong_frequency:.3f}: {sse_wrong:.4f}")
print(f"SSE at best frequency  {best_frequency:.3f}: {sse_best:.4f}")
print("Lower SSE means better fit.")

In [ ]:
# Plot the fitted model over a smooth time grid
t_smooth = np.linspace(t_irregular.min(), t_irregular.max(), 1000)

def predict_sine_cosine(t_new, coefficients, frequency):
    a, b, c = coefficients
    return (
        a * np.sin(2 * np.pi * frequency * t_new)
        + b * np.cos(2 * np.pi * frequency * t_new)
        + c
    )

fit_best_smooth = predict_sine_cosine(t_smooth, coef_best, best_frequency)
fit_wrong_smooth = predict_sine_cosine(t_smooth, coef_wrong, wrong_frequency)

plt.scatter(t_irregular, y_irregular, s=10, alpha=0.5, label="Centered data")
plt.plot(t_smooth, fit_best_smooth, linewidth=2, label="Best-frequency fit")
plt.plot(t_smooth, fit_wrong_smooth, linewidth=2, alpha=0.7, label="Wrong-frequency fit")
plt.xlabel("Time [days]")
plt.ylabel("Centered Flux")
plt.title("Sine/Cosine Regression at Different Frequencies")
plt.legend()
plt.show()

## 3.8 Manual frequency scan using least squares

Now we will scan many frequencies manually.

For each frequency:

1. build sine/cosine features
2. fit coefficients using least squares
3. compute squared error
4. choose the frequency with the smallest error

This is not the full optimized Lomb–Scargle formula, but it shows the main concept.

In [ ]:
manual_sse = []

for f in frequencies:
    _, _, sse = fit_sine_cosine_model(t_irregular, y_irregular, f)
    manual_sse.append(sse)

manual_sse = np.array(manual_sse)

manual_best_frequency = frequencies[np.argmin(manual_sse)]
manual_best_period = 1 / manual_best_frequency

plt.plot(frequencies, manual_sse)
plt.axvline(true_frequency_ls, linestyle="--", label="True frequency")
plt.axvline(manual_best_frequency, linestyle=":", label="Manual best frequency")
plt.xlabel("Frequency [cycles/day]")
plt.ylabel("Sum of Squared Errors")
plt.title("Manual Least-Squares Frequency Search")
plt.legend()
plt.show()

print(f"Manual best period: {manual_best_period:.4f} days")
print(f"Lomb-Scargle best period: {best_period:.4f} days")

# Part 4 — Connecting This to TESS Variable Star Research

## 4.1 How this relates to your TESS project

In a TESS variable-star classification project, these ideas appear naturally:

### Fourier analysis

Useful for understanding that a light curve can contain frequency components.

### Linear regression

Useful because many fitting methods, including parts of Lomb–Scargle, are based on least-squares fitting.

### Lomb–Scargle

Useful for extracting period-related features from light curves, especially when the data have gaps or uneven sampling.

Examples of possible machine-learning features:

- best Lomb–Scargle period
- maximum Lomb–Scargle power
- top 2 or top 3 periods
- ratios between top periods
- false alarm probability, if using a library that provides it
- phase-folded curve shape features

## 4.2 Important caution: period is not always perfect

The strongest Lomb–Scargle peak is not always the true physical period.

Possible problems include:

- noise
- data gaps
- long-term instrumental trends
- harmonic confusion, such as finding \(P/2\) instead of \(P\)
- aliases caused by the observing pattern
- non-sinusoidal light-curve shapes

For example, eclipsing binaries may have two dips per orbit.  
A period-finding algorithm might detect half the orbital period if the two dips look similar.

## 4.3 Practical workflow for TESS light curves

A common simplified workflow is:

1. load time and flux from FITS files
2. remove invalid values
3. optionally remove bad-quality cadences
4. normalize flux
5. optionally detrend long-term variation
6. compute Lomb–Scargle periodogram
7. extract period/frequency features
8. phase fold the light curve for visual checking
9. use features in a classifier, such as random forest

For your project, this means Lomb–Scargle is not just a math topic.  
It becomes a feature-engineering tool.

# Part 5 — Mini Exercises

Try changing the code above and answer these questions:

1. What happens if the noise level increases from `0.02` to `0.08`?
2. What happens if the time gap becomes larger?
3. What happens if the true period is longer than the observation window?
4. Can Lomb–Scargle find two different periods if the signal contains two sine waves?
5. For an eclipsing-binary-like signal, does Lomb–Scargle find the orbital period or half the orbital period?

These are not just coding exercises.  
They are close to real research questions in variable-star analysis.

# Summary

Fourier analysis says:

> Periodic signals can be decomposed into frequency components.

Linear regression says:

> We can fit model parameters by minimizing squared errors.

Lomb–Scargle combines these ideas:

> Try many frequencies, fit sine/cosine models, and find which frequency explains the light curve best.

For TESS light curves, this helps turn raw time-series data into useful astrophysical and machine-learning features.